# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj), using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# The Croissant schema JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("\033[1mDataset Title:\033[0m", metadata['name'])
print("\033[1mDescription:\033[0m", metadata['description'])

## 2. Data Overview
Review available record sets, their `@id`s, and the fields/columns available in the dataset.

We will print information about each record set and its fields. All references are to their Croissant `@id`.

In [ ]:
# Get all available record sets and display their IDs and fields
def show_record_sets_and_fields(dataset):
    print("Available RecordSets and Fields:\n-----------------------------")
    for rs in dataset.record_sets:
        rs_json = rs.to_json()
        print(f"[RecordSet] @id: {rs_json['@id']} | name: {rs_json.get('name', 'N/A')}")
        if 'field' in rs_json and isinstance(rs_json['field'], list):
            for field in rs_json['field']:
                if isinstance(field, dict):
                    field_id = field['@id']
                else:
                    field_id = field
                # Try to locate the field definition
                try:
                    fd = dataset.field(field_id)
                    fd_json = fd.to_json()
                    print(f"   \u2514 [Field] @id: {fd_json['@id']} | name: {fd_json.get('name', 'N/A')} | dataType: {fd_json.get('dataType', 'N/A')}")
                    # Show columns
                    if 'column' in fd_json:
                        if isinstance(fd_json['column'], list):
                            for col in fd_json['column']:
                                col_id = col if isinstance(col, str) else col['@id']
                                try:
                                    cobj = dataset.column(col_id)
                                    cjson = cobj.to_json()
                                    print(f"      - [Column] @id: {cjson['@id']} | name: {cjson.get('name', 'N/A')}")
                                except Exception:
                                    pass
                        else:
                            col_id = fd_json['column']
                            try:
                                cobj = dataset.column(col_id)
                                cjson = cobj.to_json()
                                print(f"      - [Column] @id: {cjson['@id']} | name: {cjson.get('name', 'N/A')}")
                            except Exception:
                                pass
                except Exception:
                    print(f"   \u2514 [Field] @id: {field_id} (definition not found)")
    print("-----------------------------\n")

show_record_sets_and_fields(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

**Note:** Replace `record_set_ids` below with the actual RecordSet `@id`s from the previous section.

In [ ]:
# Example: Extract data from found record sets
# Gather all record set IDs programmatically
record_set_ids = [rs.to_json()['@id'] for rs in dataset.record_sets]
print("Record sets being loaded:")
print(record_set_ids)

# Load all record sets into DataFrames and inspect first rows and columns
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\n[RecordSet: {record_set_id}] Columns: {df.columns.tolist()}")
    print(df.head(3))

# For demonstration, pick the first record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing: filtering, normalization, and grouping. 

> Fields are referenced by their Croissant `@id`. Please update the field IDs in the code below as needed based on overview section output.

Let's pick a key numeric field from the main record set for demonstration (e.g., a survey score field).

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"\nColumns in [{main_record_set_id}]: {df.columns.tolist()}")
    
    # Try to find a numeric field (e.g., GAD-7, PHQ-9, or PSQ score by @id or column name)
    candidate_numeric_fields = [c for c in df.columns if df[c].dtype.kind in 'iufc']
    # Fallback: Accept all fields and check
    numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.columns[0]
    print(f"Selected numeric field for example: {numeric_field}")
    
    # Filter for records with high values in numeric field (>10 as example)
    threshold = 10
    # Drop na for the filter
    fdf = df[df[numeric_field].dropna() > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold} (showing up to 5 rows):")
    print(fdf.head())

    # Normalize field
    if not fdf.empty:
        col_norm = f"{numeric_field}_normalized"
        fdf[col_norm] = (fdf[numeric_field] - fdf[numeric_field].mean()) / fdf[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(fdf[[numeric_field, col_norm]].head())
    
    # Try grouping by a likely categorical field (e.g., 'gender' or similar)
    group_candidates = [c for c in df.columns if c.lower() in ['gender', 'sex', 'village', 'marital_status', 'level_of_education']]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped = fdf.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped)
    else:
        print("No obvious grouping field found.")
else:
    print('No record sets detected in this dataset.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset with simple plots.

Below: We demonstrate a histogram of the main numeric field (if available) and a boxplot by group, if such field is present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not df.empty:
    field_to_plot = numeric_field
    plt.figure(figsize=(7,4))
    df[field_to_plot].dropna().hist(bins=15, color="skyblue", edgecolor="k")
    plt.title(f"Histogram of {field_to_plot}")
    plt.xlabel(field_to_plot)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by a group if available
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=field_to_plot)
        plt.title(f"{field_to_plot} distribution by {group_field}")
        plt.ylabel(field_to_plot)
        plt.xlabel(group_field)
        plt.show()


## 6. Conclusion

In this notebook, we demonstrated how to explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` Python library. We programmatically identified available record sets and fields by their `@id`, loaded records into Pandas DataFrames, and performed basic EDA and visualization steps. All Croissant entities were referenced by their `@id` for traceability. 

This notebook template can be adapted to other Croissant datasets by using the proper schema URL and updating record set and field IDs as required.
